# BIST Hisseleri ve TCMB Döviz Kuru Verileriyle Risk-Getiri Analizi

## 1. Proje Tanımı

Bu notebook, seçilmiş BIST hisseleri ve TCMB USD/TRY kuru verileri üzerinde yapılacak veri bilimi dönem projesinin ana çalışma dosyasıdır.

Analiz günlük getiri, volatilite/risk, korelasyon ve basit regresyon modeline odaklanacaktır.

Notebook adım adım geliştirilecektir. Bu aşamada notebook iskeletine ham veri yükleme akışı eklenmiştir.

Proje eğitim amaçlıdır ve yatırım tavsiyesi değildir.

## 2. Veri Kaynakları ve Kullanılan Finansal Varlıklar

Projede seçilen BIST hisseleri ile TCMB USD/TRY döviz kuru verilerinin kullanılması planlanmaktadır.

Planlanan BIST hisseleri: THYAO.IS, ASELS.IS, GARAN.IS, SISE.IS ve KCHOL.IS.

Döviz kuru değişkeni olarak TCMB USD/TRY kuru kullanılacaktır.

## 3. Kütüphanelerin Yüklenmesi

Bu bölümde proje boyunca kullanılacak temel veri analizi ve görselleştirme kütüphaneleri yüklenecektir.

Bu aşamada temel kütüphanelere ek olarak BIST verisini indirmek için `yfinance` ve proje yollarını yönetmek için `pathlib` eklenmiştir. Modelleme kütüphaneleri ileriki adımlarda eklenecektir.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from pathlib import Path

## 4. Veri Yükleme Planı

Bu bölümde yalnızca ham veri yükleme akışı hazırlanacaktır.

BIST hisse fiyatları Yahoo Finance üzerinden `yfinance` kütüphanesi ile indirilecektir. TCMB USD/TRY kuru verisi ise TCMB kaynağından alınmış yerel bir CSV dosyası olarak beklenmektedir.

Bu adımda veri temizleme, veri birleştirme, günlük getiri hesaplama, grafik oluşturma veya modelleme yapılmayacaktır. Bu işlemler ileriki adımlarda ele alınacaktır.

In [ ]:
current_path = Path.cwd().resolve()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "2022-01-01"
END_DATE = "2026-06-30"
BIST_TICKERS = ["THYAO.IS", "ASELS.IS", "GARAN.IS", "SISE.IS", "KCHOL.IS"]

print(f"Proje ana klasörü: {PROJECT_ROOT}")
print(f"Ham veri klasörü: {RAW_DATA_DIR}")

### BIST Hisse Verilerinin Yüklenmesi

Seçilen BIST hisselerinin fiyat verileri Yahoo Finance üzerinden indirilecektir. Bu aşamada yalnızca kapanış fiyatları seçilecek ve ham veri olarak `data/raw/bist_close_prices_raw.csv` dosyasına kaydedilecektir.

Bu bölümde günlük getiri hesaplaması veya görselleştirme yapılmayacaktır.

In [ ]:
yfinance_end_date = (pd.to_datetime(END_DATE) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

bist_raw = yf.download(
    tickers=BIST_TICKERS,
    start=START_DATE,
    end=yfinance_end_date,
    auto_adjust=True,
    progress=False
)

if isinstance(bist_raw.columns, pd.MultiIndex):
    if "Close" in bist_raw.columns.get_level_values(0):
        bist_close = bist_raw["Close"].copy()
    elif "Close" in bist_raw.columns.get_level_values(1):
        bist_close = bist_raw.xs("Close", axis=1, level=1).copy()
    else:
        raise ValueError("İndirilen BIST verisinde kapanış fiyatı sütunu bulunamadı.")
else:
    bist_close = bist_raw[["Close"]].copy()

bist_close.index.name = "Date"
bist_close_path = RAW_DATA_DIR / "bist_close_prices_raw.csv"
bist_close.to_csv(bist_close_path)

print(f"BIST kapanış fiyatları kaydedildi: {bist_close_path}")
print(f"BIST veri boyutu: {bist_close.shape}")
bist_close.head()

### TCMB USD/TRY Verisinin Yüklenmesi

Döviz kuru verisinin TCMB kaynağından alınmış yerel bir CSV dosyası olarak `data/raw/tcmb_usdtry.csv` konumunda bulunması beklenmektedir.

CSV dosyasında tarih bilgisini içeren bir sütun ve USD/TRY kur değerini içeren bir değer sütunu olması beklenmektedir.

Bu adımda yalnızca dosyanın yüklenip yüklenemediği kontrol edilecektir. Sütun adlarının standartlaştırılması, eksik değer kontrolleri, BIST verisiyle birleştirme ve günlük getiri hesaplama işlemleri sonraki adımlarda yapılacaktır.

In [ ]:
tcmb_path = RAW_DATA_DIR / "tcmb_usdtry.csv"

if tcmb_path.exists():
    tcmb_raw = pd.read_csv(tcmb_path)
    print(f"TCMB USD/TRY ham verisi yüklendi: {tcmb_path}")
    print(f"TCMB veri boyutu: {tcmb_raw.shape}")
else:
    print("TCMB USD/TRY CSV dosyası bulunamadı.")
    print(f"Lütfen dosyayı şu konuma yerleştirin: {tcmb_path}")
    tcmb_raw = pd.DataFrame()

tcmb_raw.head()

## 5. Veri Temizleme ve Birleştirme

Bu adımda ham BIST ve TCMB verileri analiz için hazırlanır.

Tarih formatları standartlaştırılır; eksik değerler, veri tipleri ve tekrar eden tarihler kontrol edilir.

TCMB USD/TRY döviz alış kuru, `USDTRY_TCMB` değişkeni olarak kullanılacaktır.

BIST ve TCMB verileri ortak tarihler üzerinden birleştirilecektir. Günlük getiri hesaplama bir sonraki adımda yapılacaktır.

### BIST Ham Kapanış Fiyatlarının Yüklenmesi

Bu hücre, BIST kapanış fiyatları önceki hücrelerde zaten oluşturulduysa o veriyi kullanır. Eğer veri bellekte yoksa `data/raw/bist_close_prices_raw.csv` dosyasından yüklemeyi dener.

In [ ]:
bist_raw_path = RAW_DATA_DIR / "bist_close_prices_raw.csv"

if "bist_close_prices" in globals():
    bist_source = bist_close_prices.copy()
    print("BIST kapanış fiyatları önceki hücrelerdeki bist_close_prices değişkeninden alındı.")
elif "bist_close" in globals():
    bist_source = bist_close.copy()
    print("BIST kapanış fiyatları önceki yfinance hücresindeki bist_close değişkeninden alındı.")
elif bist_raw_path.exists():
    bist_source = pd.read_csv(bist_raw_path)
    print(f"BIST kapanış fiyatları ham CSV dosyasından yüklendi: {bist_raw_path}")
else:
    raise FileNotFoundError(
        "BIST ham kapanış fiyatları bulunamadı. "
        "Önce yfinance veri yükleme hücresini çalıştırın veya "
        "data/raw/bist_close_prices_raw.csv dosyasını oluşturun."
    )

print(f"BIST ham veri boyutu: {bist_source.shape}")
bist_source.head()

### BIST Verisinin Temizlenmesi

Bu bölümde BIST kapanış fiyatlarının tarih alanı standartlaştırılır, seçilen hisse sütunları korunur ve temel kalite kontrolleri yapılır.

Eksik değerler, veri tipleri ve tekrar eden tarihler kontrol edilir.

In [ ]:
selected_bist_columns = BIST_TICKERS
bist_clean = bist_source.copy()

if isinstance(bist_clean.columns, pd.MultiIndex):
    if "Close" in bist_clean.columns.get_level_values(0):
        bist_clean = bist_clean["Close"].copy()
    elif "Close" in bist_clean.columns.get_level_values(1):
        bist_clean = bist_clean.xs("Close", axis=1, level=1).copy()
    else:
        raise ValueError("BIST verisinde kapanış fiyatı alanı bulunamadı.")

bist_clean.columns = [str(column).strip() for column in bist_clean.columns]
date_candidates = [
    column for column in bist_clean.columns
    if column.lower() in ["date", "datetime"] or column.lower().startswith("unnamed")
]

if date_candidates:
    bist_clean = bist_clean.rename(columns={date_candidates[0]: "date"})
else:
    bist_clean = bist_clean.reset_index()
    bist_clean = bist_clean.rename(columns={bist_clean.columns[0]: "date"})

bist_clean["date"] = pd.to_datetime(bist_clean["date"], errors="coerce")

if bist_clean["date"].isna().any():
    raise ValueError("BIST tarih alanında dönüştürülemeyen değerler var.")

missing_bist_columns = [column for column in selected_bist_columns if column not in bist_clean.columns]
if missing_bist_columns:
    raise ValueError(f"BIST verisinde beklenen sütunlar eksik: {missing_bist_columns}")

for column in selected_bist_columns:
    bist_clean[column] = pd.to_numeric(bist_clean[column], errors="coerce")

bist_clean = bist_clean[["date"] + selected_bist_columns]
bist_clean = bist_clean.sort_values("date").reset_index(drop=True)

duplicate_bist_dates = bist_clean["date"].duplicated().sum()
print(f"BIST tekrar eden tarih sayısı: {duplicate_bist_dates}")

if duplicate_bist_dates > 0:
    bist_clean = bist_clean.drop_duplicates(subset="date", keep="first").reset_index(drop=True)
    print("BIST verisinde tekrar eden tarihler ilk kayıt korunarak kaldırıldı.")

print(f"BIST temiz veri boyutu: {bist_clean.shape}")
print("BIST eksik değer sayıları:")
print(bist_clean.isna().sum())
print("\nBIST veri tipleri:")
print(bist_clean.dtypes)

bist_clean.head()

### TCMB USD/TRY Verisinin Temizlenmesi

Bu bölümde TCMB ham döviz kuru verisi yüklenir ve analizde yalnızca `date` ile `usdtry_forex_buying` sütunları kullanılır.

`usdtry_forex_buying` sayısal formata çevrilir ve `USDTRY_TCMB` adıyla yeniden adlandırılır.

In [ ]:
tcmb_raw_path = RAW_DATA_DIR / "tcmb_usdtry.csv"

if not tcmb_raw_path.exists():
    raise FileNotFoundError(
        "TCMB USD/TRY ham veri dosyası bulunamadı. "
        "Lütfen data/raw/tcmb_usdtry.csv dosyasını ekleyin."
    )

tcmb_source = pd.read_csv(tcmb_raw_path)
tcmb_clean = tcmb_source.copy()

required_tcmb_columns = ["date", "usdtry_forex_buying"]
missing_tcmb_columns = [column for column in required_tcmb_columns if column not in tcmb_clean.columns]
if missing_tcmb_columns:
    raise ValueError(f"TCMB verisinde beklenen sütunlar eksik: {missing_tcmb_columns}")

tcmb_clean["date"] = pd.to_datetime(tcmb_clean["date"], errors="coerce")
tcmb_clean["USDTRY_TCMB"] = pd.to_numeric(tcmb_clean["usdtry_forex_buying"], errors="coerce")
tcmb_clean = tcmb_clean[["date", "USDTRY_TCMB"]]
tcmb_clean = tcmb_clean.sort_values("date").reset_index(drop=True)

if tcmb_clean["date"].isna().any():
    raise ValueError("TCMB tarih alanında dönüştürülemeyen değerler var.")

duplicate_tcmb_dates = tcmb_clean["date"].duplicated().sum()
print(f"TCMB tekrar eden tarih sayısı: {duplicate_tcmb_dates}")

if duplicate_tcmb_dates > 0:
    tcmb_clean = tcmb_clean.drop_duplicates(subset="date", keep="first").reset_index(drop=True)
    print("TCMB verisinde tekrar eden tarihler ilk kayıt korunarak kaldırıldı.")

print(f"TCMB temiz veri boyutu: {tcmb_clean.shape}")
print("TCMB eksik değer sayıları:")
print(tcmb_clean.isna().sum())
print("\nTCMB veri tipleri:")
print(tcmb_clean.dtypes)

tcmb_clean.head()

### BIST ve TCMB Verilerinin Birleştirilmesi

BIST ve TCMB verileri `date` sütunu üzerinden iç birleşim yöntemiyle birleştirilecektir.

Bu yöntemle yalnızca iki veri kaynağında da ortak olan işlem günleri analiz veri setine alınır.

In [ ]:
price_data = pd.merge(
    bist_clean,
    tcmb_clean,
    on="date",
    how="inner"
)

price_data = price_data.sort_values("date").reset_index(drop=True)

print(f"Birleştirilmiş fiyat veri seti boyutu: {price_data.shape}")
print(f"Başlangıç tarihi: {price_data['date'].min()}")
print(f"Bitiş tarihi: {price_data['date'].max()}")
print("\nBirleştirilmiş veri eksik değer sayıları:")
print(price_data.isna().sum())
print("\nBirleştirilmiş veri tipleri:")
print(price_data.dtypes)

print("\nİlk satırlar:")
display(price_data.head())

print("Son satırlar:")
display(price_data.tail())

print("Sayısal değişkenlerin özet istatistikleri:")
display(price_data.describe())

### Temizlenmiş Fiyat Verisinin Kaydedilmesi

Ortak tarihler üzerinden oluşturulan fiyat düzeyindeki veri seti `data/processed/clean_price_data.csv` dosyasına kaydedilecektir.

Bu dosya günlük getiri hesaplama adımında kullanılacaktır.

In [ ]:
clean_price_data_path = PROCESSED_DATA_DIR / "clean_price_data.csv"
price_data.to_csv(clean_price_data_path, index=False)

print(f"Temizlenmiş fiyat veri seti kaydedildi: {clean_price_data_path}")

## 6. Günlük Getiri Hesaplama

Bu adımda fiyat seviyelerinden günlük yüzde getiriler hesaplanır.

Ham fiyatlar farklı ölçeklerde olduğu için varlıkları doğrudan fiyatlarla karşılaştırmak yanıltıcı olabilir. Günlük getiriler, hisseleri ve döviz kurunu daha karşılaştırılabilir hale getirir.

EDA ve görselleştirmeler bir sonraki adımda yapılacaktır. Modelleme bu adımda yapılmayacaktır.

### Temizlenmiş Fiyat Verisinin Yüklenmesi

Bu hücre, bir önceki adımda oluşturulan `price_data` değişkeni bellekte varsa onu kullanır. Eğer bellekte yoksa `data/processed/clean_price_data.csv` dosyasından yüklemeyi dener.

Dosya bulunamazsa önceki veri temizleme ve birleştirme hücrelerinin çalıştırılması gerektiğini belirten Türkçe bir mesaj gösterilir.

In [ ]:
clean_price_data_path = PROCESSED_DATA_DIR / "clean_price_data.csv"

if "price_data" in globals() and isinstance(price_data, pd.DataFrame) and not price_data.empty:
    price_source = price_data.copy()
    print("Temizlenmiş fiyat verisi önceki hücrelerdeki price_data değişkeninden alındı.")
elif clean_price_data_path.exists():
    price_source = pd.read_csv(clean_price_data_path)
    print(f"Temizlenmiş fiyat verisi dosyadan yüklendi: {clean_price_data_path}")
else:
    print("Temizlenmiş fiyat veri dosyası bulunamadı.")
    print("Lütfen önce veri temizleme ve birleştirme bölümündeki hücreleri çalıştırın.")
    price_source = pd.DataFrame()

print(f"Fiyat verisi boyutu: {price_source.shape}")
price_source.head()

### Getiri Hesaplaması İçin Verinin Hazırlanması

Bu bölümde tarih alanı standartlaştırılır, veri tarihe göre sıralanır ve günlük getiri hesaplamasında kullanılacak fiyat sütunları seçilir.

In [ ]:
price_columns = ["THYAO.IS", "ASELS.IS", "GARAN.IS", "SISE.IS", "KCHOL.IS", "USDTRY_TCMB"]
required_price_columns = ["date"] + price_columns

if price_source.empty:
    price_for_returns = pd.DataFrame(columns=required_price_columns)
    print("Fiyat verisi boş olduğu için getiri hesaplamasına hazırlık yapılamadı.")
else:
    missing_price_columns = [column for column in required_price_columns if column not in price_source.columns]
    if missing_price_columns:
        raise ValueError(f"Temiz fiyat verisinde beklenen sütunlar eksik: {missing_price_columns}")

    price_for_returns = price_source[required_price_columns].copy()
    price_for_returns["date"] = pd.to_datetime(price_for_returns["date"], errors="coerce")

    if price_for_returns["date"].isna().any():
        raise ValueError("Fiyat verisindeki tarih alanında dönüştürülemeyen değerler var.")

    for column in price_columns:
        price_for_returns[column] = pd.to_numeric(price_for_returns[column], errors="coerce")

    price_for_returns = price_for_returns.sort_values("date").reset_index(drop=True)

print(f"Getiri hesaplamasına hazırlanmış fiyat verisi boyutu: {price_for_returns.shape}")
price_for_returns.head()

### Günlük Yüzde Getirilerin Hesaplanması

Bu bölümde BIST hisseleri ve TCMB USD/TRY kuru için günlük yüzde getiriler `pct_change()` ile hesaplanır.

İlk satır, önceki gün bilgisi olmadığı için eksik değer içerir ve veri setinden çıkarılır.

In [ ]:
return_column_names = {
    "THYAO.IS": "THYAO_Return",
    "ASELS.IS": "ASELS_Return",
    "GARAN.IS": "GARAN_Return",
    "SISE.IS": "SISE_Return",
    "KCHOL.IS": "KCHOL_Return",
    "USDTRY_TCMB": "USDTRY_Return",
}

if price_for_returns.empty:
    return_data = pd.DataFrame(columns=["date"] + list(return_column_names.values()))
    print("Fiyat verisi boş olduğu için günlük getiri hesaplanmadı.")
else:
    # Ham fiyatlar farklı ölçeklerde olduğu için günlük getiriler varlıkları karşılaştırmayı kolaylaştırır.
    calculated_returns = price_for_returns[price_columns].pct_change()
    calculated_returns = calculated_returns.rename(columns=return_column_names)

    return_data = pd.concat(
        [price_for_returns[["date"]], calculated_returns],
        axis=1
    )
    return_data = return_data.dropna().reset_index(drop=True)

print(f"Günlük getiri veri seti boyutu: {return_data.shape}")
return_data.head()

### Günlük Getiri Verisi Kontrolleri

Günlük getiri veri seti oluşturulduktan sonra ilk ve son satırlar, veri boyutu, eksik değerler, veri tipleri ve temel özet istatistikler kontrol edilir.

Bu adımda grafik oluşturulmaz ve araştırma soruları yanıtlanmaz.

In [ ]:
print(f"Günlük getiri veri seti boyutu: {return_data.shape}")

print("İlk satırlar:")
display(return_data.head())

print("Son satırlar:")
display(return_data.tail())

print("Eksik değer sayıları:")
print(return_data.isna().sum())

print("\nVeri tipleri:")
print(return_data.dtypes)

print("\nGünlük getiriler için özet istatistikler:")
display(return_data.describe())

### Günlük Getiri Verisinin Kaydedilmesi

Oluşturulan günlük getiri veri seti `data/processed/daily_return_data.csv` dosyasına kaydedilecektir.

Bu dosya bir sonraki EDA ve görselleştirme adımlarında kullanılacaktır.

In [ ]:
daily_return_data_path = PROCESSED_DATA_DIR / "daily_return_data.csv"

if return_data.empty:
    print("Günlük getiri veri seti boş olduğu için dosya kaydedilmedi.")
else:
    return_data.to_csv(daily_return_data_path, index=False)
    print(f"Günlük getiri veri seti kaydedildi: {daily_return_data_path}")

## 7. Keşifsel Veri Analizi Planı

Bu bölümde temel istatistikler, dağılımlar, volatilite karşılaştırmaları ve değişkenler arasındaki ilişkiler incelenecektir.

EDA aşaması, araştırma sorularını yanıtlamaya yardımcı olacak şekilde sade tutulacaktır.

## 8. Görselleştirme Planı

Bu bölümde zaman serisi grafikleri, dağılım grafikleri, korelasyon ısı haritası ve risk-getiri karşılaştırmaları gibi görselleştirmeler planlanmaktadır.

Bu aşamada grafik oluşturulmamıştır.

## 9. Araştırma Soruları

Bu projede aşağıdaki araştırma sorularının yanıtlanması planlanmaktadır:

1. Seçilen BIST hisselerinin günlük getiri dağılımları nasıldır?
2. Hangi hisse daha yüksek risk/volatilite göstermektedir?
3. TCMB USD/TRY kuru ile BIST hisselerinin günlük getirileri arasında ilişki var mıdır?
4. Seçilen BIST hisseleri birbirleriyle benzer hareket ediyor mu?

## 10. Basit Modelleme Planı

Bu bölümde ileriki aşamada basit ve açıklanabilir bir regresyon modeli kurulması planlanmaktadır.

Modelleme adımı henüz uygulanmamıştır. Bu aşamada scikit-learn import edilmemiştir.

## 11. Bulguların Yorumlanması

Analiz tamamlandıktan sonra elde edilen bulgular finans ve ekonomi teması bağlamında yorumlanacaktır.

Yorumlar, model sonuçlarının ve görselleştirmelerin sınırlılıkları dikkate alınarak yapılacaktır.

## 12. Sınırlamalar

Finansal günlük getiriler gürültülü olabilir ve kısa vadeli tahminlerde yüksek doğruluk beklenmemelidir.

Bu proje eğitim amaçlıdır ve yatırım tavsiyesi değildir. Kullanılacak model basit ve açıklanabilir tutulacaktır.

## 13. Sonuç

Bu notebook iskeleti, projenin sonraki veri yükleme, temizleme, analiz, görselleştirme ve modelleme adımları için temel yapıyı oluşturur.

İleriki adımlarda her bölüm sırayla geliştirilecektir.